# Architecture 2 — GRU Baseline

Abductive Commonsense Reasoning using frozen fastText
embeddings and a two-layer GRU.

This notebook implements and evaluates the baseline.
Hyperparameter optimization is performed separately.

In [6]:
import sys
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset

# Project paths
repo_root = Path.cwd().parents[1]
sys.path.insert(0, str(repo_root))

from project1.src.models import EmbeddingGRU
from project1.src.data_loader import create_dataloader
from project1.src.training import train_one_epoch, evaluate

# Reproducibility
seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("=== COMPUTE DEVICE ===")
print("Device:", device)
print("PyTorch:", torch.__version__)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
    
# Hyperparameters
config = {
    "architecture": "GRU",
    "hidden_dim": 128,
    "num_layers": 2,
    "dropout": 0.2,
    "learning_rate": 0.0003,
    "batch_size": 32,
    "max_epochs": 60,
    "patience": 15,
    "seed": seed
}

# Load training and validation splits only
train_data = load_dataset(
    "amarfurt/art-nlp-hs26",
    split="train"
)

val_data = load_dataset(
    "amarfurt/art-nlp-hs26",
    split="validation"
)

# Load verified preprocessing artifacts
artifact_path = (
    repo_root / "project1" / "artifacts" / "fasttext_art.pt"
)

saved = torch.load(
    artifact_path,
    map_location="cpu",
    weights_only=True
)

vocab = saved["vocab"]
embedding_weights = saved["embedding_weights"]

assert len(vocab) == 15038
assert embedding_weights.shape == (15038, 300)

# Frozen fastText embedding layer
embedding_layer = nn.Embedding.from_pretrained(
    embedding_weights,
    freeze=True,
    padding_idx=vocab["[PAD]"]
)

# DataLoaders
train_loader = create_dataloader(
    train_data,
    vocab,
    batch_size=config["batch_size"],
    shuffle=True
)

val_loader = create_dataloader(
    val_data,
    vocab,
    batch_size=config["batch_size"],
    shuffle=False
)

# Initialize GRU
model = EmbeddingGRU(
    embedding_layer=embedding_layer,
    pad_id=vocab["[PAD]"],
    hidden_dim=config["hidden_dim"],
    num_layers=config["num_layers"],
    dropout=config["dropout"]
).to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    [p for p in model.parameters() if p.requires_grad],
    lr=config["learning_rate"]
)

# Verification
print("\n=== GRU SETUP ===")
print("Architecture:", config["architecture"])
print("GRU layers:", config["num_layers"])
print("Hidden dimension:", config["hidden_dim"])
print("Dropout:", config["dropout"])
print("Learning rate:", config["learning_rate"])
print("Batch size:", config["batch_size"])

print("\nVocabulary size:", len(vocab))
print("Embedding shape:", tuple(embedding_weights.shape))
print("Embeddings frozen:",
      not model.embedding.weight.requires_grad)

print("\nTraining examples:", len(train_data))
print("Validation examples:", len(val_data))

print("Trainable parameters:", sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
))

# Final model checks
assert model.embedding.weight.requires_grad is False
assert next(model.gru.parameters()).device == device

batch = next(iter(val_loader))

model.eval()

with torch.inference_mode():
    logits = model(
        batch["input_ids"].to(device),
        lengths=batch["lengths"]
    )

assert logits.shape == (batch["labels"].size(0), 2)
assert torch.isfinite(logits).all().item()

print("\nModel setup verification: PASS")

=== COMPUTE DEVICE ===
Device: cuda
PyTorch: 2.14.1+cu130
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
CUDA: 13.0

=== GRU SETUP ===
Architecture: GRU
GRU layers: 2
Hidden dimension: 128
Dropout: 0.2
Learning rate: 0.0003
Batch size: 32

Vocabulary size: 15038
Embedding shape: (15038, 300)
Embeddings frozen: True

Training examples: 16801
Validation examples: 1000
Trainable parameters: 280962


In [3]:
import wandb
from pathlib import Path

# Checkpoint directory
checkpoint_dir = (
    repo_root / "project1" / "artifacts"
)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Descriptive experiment name
run_name = (
    f"gru_h{config['hidden_dim']}"
    f"_l{config['num_layers']}"
    f"_lr{config['learning_rate']}"
    f"_bs{config['batch_size']}"
    f"_drop{config['dropout']}"
    f"_seed{config['seed']}_baseline"
)

# Initialize W&B
run = wandb.init(
    entity="kimbortis-hochschule-luzern",
    project="nlp-art-hs26-project1",
    group="fasttext-gru",
    job_type="train",
    name=run_name,
    config={
        **config,
        "optimizer": "Adam",
        "weight_decay": 0.0,
        "embedding": "fasttext-wiki-news-subwords-300",
        "embedding_frozen": True,
        "vocab_size": len(vocab),
        "preprocessing": "v2",
        "device": str(device),
        "gpu_name": (
            torch.cuda.get_device_name(0)
            if device.type == "cuda"
            else None
        ),
    }
)

# Use epoch as X-axis
run.define_metric("epoch")
run.define_metric("train/*", step_metric="epoch")
run.define_metric("val/*", step_metric="epoch")

checkpoint_path = (
    checkpoint_dir / f"{run_name}_{run.id}_best.pt"
)

best_val_accuracy = -1.0
best_val_loss = float("inf")
lowest_val_loss = float("inf")

best_epoch = 0
no_improvement = 0

print("W&B run:", run.name)
print("W&B URL:", run.url)
print("Checkpoint:", checkpoint_path)

try:
    for epoch in range(1, config["max_epochs"] + 1):

        # Training
        train_metrics = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device
        )

        # Validation
        val_metrics = evaluate(
            model,
            val_loader,
            criterion,
            device
        )

        # W&B logging
        run.log({
            "epoch": epoch,
            "train/loss": train_metrics["loss"],
            "train/accuracy": train_metrics["accuracy"],
            "val/loss": val_metrics["loss"],
            "val/accuracy": val_metrics["accuracy"]
        })

        # Track lowest validation loss separately
        lowest_val_loss = min(
            lowest_val_loss,
            val_metrics["loss"]
        )

        print(
            f"Epoch {epoch:02d} | "
            f"Train loss: {train_metrics['loss']:.4f} | "
            f"Train acc: {train_metrics['accuracy']:.2%} | "
            f"Val loss: {val_metrics['loss']:.4f} | "
            f"Val acc: {val_metrics['accuracy']:.2%}"
        )

        # Checkpoint selection:
        # best accuracy, with loss as tiebreaker
        improved = (
            val_metrics["accuracy"] > best_val_accuracy
            or (
                val_metrics["accuracy"] == best_val_accuracy
                and val_metrics["loss"] < best_val_loss
            )
        )

        if improved:
            best_val_accuracy = val_metrics["accuracy"]
            best_val_loss = val_metrics["loss"]
            best_epoch = epoch
            no_improvement = 0

            torch.save(
                model.state_dict(),
                checkpoint_path
            )
        else:
            no_improvement += 1

        # Early stopping
        if no_improvement >= config["patience"]:
            print(f"\nEarly stopping at epoch {epoch}")
            break

    # Store best results in W&B
    run.summary["best_epoch"] = best_epoch
    run.summary["best_val_accuracy"] = best_val_accuracy
    run.summary["val_loss_at_best_accuracy"] = best_val_loss
    run.summary["lowest_val_loss"] = lowest_val_loss
    run.summary["epochs_trained"] = epoch

    print("\n=== BEST GRU MODEL ===")
    print("Best epoch:", best_epoch)
    print(f"Best validation accuracy: {best_val_accuracy:.2%}")
    print(f"Validation loss at best: {best_val_loss:.4f}")
    print(f"Lowest validation loss: {lowest_val_loss:.4f}")
    print("Checkpoint:", checkpoint_path)

finally:
    run.finish()


wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from C:\Users\kimbo\_netrc.
wandb: Currently logged in as: kimbortis (kimbortis-hochschule-luzern) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


W&B run: gru_h128_l2_lr0.0003_bs32_drop0.2_seed42_baseline
W&B URL: https://forge.coreweave.com/wandb/kimbortis-hochschule-luzern/nlp-art-hs26-project1/runs/2v3bk3hl
Checkpoint: C:\Users\kimbo\Documents\HSLU\Sem 5\NLP\nlp-art-hs26-kim\project1\artifacts\gru_h128_l2_lr0.0003_bs32_drop0.2_seed42_baseline_2v3bk3hl_best.pt
Epoch 01 | Train loss: 0.6933 | Train acc: 50.49% | Val loss: 0.6928 | Val acc: 52.30%
Epoch 02 | Train loss: 0.6835 | Train acc: 55.25% | Val loss: 0.6589 | Val acc: 60.10%
Epoch 03 | Train loss: 0.6388 | Train acc: 63.10% | Val loss: 0.6417 | Val acc: 61.80%
Epoch 04 | Train loss: 0.6127 | Train acc: 65.66% | Val loss: 0.6300 | Val acc: 63.60%
Epoch 05 | Train loss: 0.6022 | Train acc: 67.09% | Val loss: 0.6193 | Val acc: 63.40%
Epoch 06 | Train loss: 0.5917 | Train acc: 68.17% | Val loss: 0.6115 | Val acc: 64.70%
Epoch 07 | Train loss: 0.5835 | Train acc: 68.85% | Val loss: 0.6097 | Val acc: 66.20%
Epoch 08 | Train loss: 0.5772 | Train acc: 69.39% | Val loss: 0.6094 |

epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
train/accuracy,▁▂▄▄▄▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇████
train/loss,██▇▆▆▆▆▆▆▅▅▅▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▁▁▁
val/accuracy,▁▄▅▆▆▆▇▇▆▇▇▇▇▇▇▇▇▇▇█▇▇▇█▇▇▇▇▇▇▇▇▇▇▇
val/loss,▄▃▂▂▁▁▁▁▁▁▁▁▁▁▂▁▁▂▁▂▁▂▂▃▃▃▄▅▄▆▅▆▇█▇
best_epoch,20
best_val_accuracy,0.681
epoch,35
epochs_trained,35
lowest_val_loss,0.60891
train/accuracy,0.85263


In [4]:
# Load the best GRU checkpoint
model.load_state_dict(
    torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=True
    )
)

model.eval()

# Evaluate restored checkpoint
restored_metrics = evaluate(
    model,
    val_loader,
    criterion,
    device
)

print("=== RESTORED GRU CHECKPOINT ===")
print("Best epoch:", best_epoch)
print(
    f"Validation accuracy: "
    f"{restored_metrics['accuracy']:.2%}"
)
print(
    f"Validation loss: "
    f"{restored_metrics['loss']:.4f}"
)

assert abs(
    restored_metrics["accuracy"] - best_val_accuracy
) < 1e-6

print("Checkpoint verification: PASS")

# Compare against provisional best MLP
mlp_best_accuracy = 0.65

difference = (
    restored_metrics["accuracy"] - mlp_best_accuracy
) * 100

print("\n=== ARCHITECTURE COMPARISON ===")
print(f"MLP best validation accuracy: {mlp_best_accuracy:.2%}")
print(
    f"GRU best validation accuracy: "
    f"{restored_metrics['accuracy']:.2%}"
)
print(f"Difference: {difference:+.2f} percentage points")


=== RESTORED GRU CHECKPOINT ===
Best epoch: 20
Validation accuracy: 68.10%
Validation loss: 0.6241
Checkpoint verification: PASS

=== ARCHITECTURE COMPARISON ===
MLP best validation accuracy: 65.00%
GRU best validation accuracy: 68.10%
Difference: +3.10 percentage points



## GRU Baseline Results

### Architecture
- Frozen fastText embeddings (300d)
- Two-layer unidirectional GRU
- Hidden dimension: 128
- Dropout: 0.2
- Trainable parameters: 280,962

### Training
- Adam, learning rate 0.0003
- Batch size: 32
- Seed: 42
- Maximum epochs: 60
- Early stopping patience: 15
- Best checkpoint selected by validation accuracy

### Results

| Metric | Result |
|---|---:|
| Best validation accuracy | 68.10% |
| Best epoch | 20 |
| Validation loss at best epoch | 0.6241 |
| Lowest validation loss | 0.6089 |
| Early stopping epoch | 35 |
| Best MLP validation accuracy | 65.00% |

### Interpretation

The GRU achieved 3.10 percentage points higher
validation accuracy than the best MLP.

However, overfitting was observed: training accuracy
continued increasing while validation loss worsened.

The baseline results were obtained on CPU. CUDA was
subsequently configured and verified on the NVIDIA
RTX 4050.

The GRU baseline has not yet been retrained on GPU.
The test split has not been evaluated.

### Experiment tracking

[W&B GRU baseline](https://forge.coreweave.com/wandb/kimbortis-hochschule-luzern/nlp-art-hs26-project1/runs/2v3bk3hl)
